Baseline RAG Model and Inference Function

In [ ]:
# import libraries for testing the setup
import sys
import torch
import datasets
import hnswlib

In [ ]:
# Test the setup
print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

In [ ]:
# Baseline RAG Model

# Import libraries 
from transformers import AutoTokenizer
from huggingface_hub import login
from mamba_ssm import MambaLMHeadModel
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel

In [ ]:
from typing import List, Optional, Callable
import numpy as np

In [ ]:
# Initialize the model and tokenizer
model_id = "state-spaces/mamba3-siso-1.5b"
tokenizer_id = "meta-llama/Llama-3.1-8B"

In [ ]:
# Run once per environment. Enter a Hugging Face token with read access.
# login()

In [ ]:
# Load thetokenizer
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_id,
    token=True,
)

# Suppress the BPE clean_up warning
tokenizer.clean_up_tokenization_spaces = False

In [ ]:
# Load the model
model = MambaLMHeadModel.from_pretrained(
    model_id,
    device="cuda",
    dtype=torch.bfloat16,
)

In [ ]:
# Set the model to evaluation mode
print("Setting the model to evaluation mode...")
model.eval()

In [ ]:
def generate_text(
    prompt: str,
    model,
    tokenizer,
    max_new_tokens: int = 64,
    temperature: float = 0.7,
    top_k: int = 50,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
    do_sample: bool = True,
) -> str:
    """
    Run inference on the baseline Mamba-3 model.
    
    Args:
        prompt: Input text prompt.
        model: Loaded MambaLMHeadModel (already on CUDA, in eval mode).
        tokenizer: Llama-3.1-8B tokenizer.
        max_new_tokens: How many new tokens to generate.
        temperature: Sampling temperature (1.0 = no change, lower = more deterministic).
        top_k / top_p: Nucleus / top-k sampling.
        repetition_penalty: >1.0 discourages repeating the same tokens.
        do_sample: If False, falls back to greedy decoding (top_k=1).
    
    Returns:
        The full generated string (prompt + continuation).
    """
    # Tokenize and move to the same device as the model
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(model.device if hasattr(model, "device") else "cuda")
    
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be non-negative")
    if do_sample and temperature <= 0:
        raise ValueError("temperature must be greater than zero when sampling")
    if top_k < 0:
        raise ValueError("top_k must be non-negative")
    if not 0 <= top_p <= 1:
        raise ValueError("top_p must be between 0 and 1")
    if repetition_penalty <= 0:
        raise ValueError("repetition_penalty must be greater than zero")

    # mamba_ssm's generic generate() uses an inference step path that currently
    # passes 3-D hidden states to Mamba3.step(), which expects 2-D states.
    # Recompute the sequence at each step to stay on the supported forward path.
    with torch.inference_mode():
        for _ in range(max_new_tokens):
            logits = model(input_ids, num_last_tokens=1).logits[:, -1, :]
            if repetition_penalty != 1.0:
                token_scores = torch.gather(logits, 1, input_ids)
                token_scores = torch.where(
                    token_scores < 0,
                    token_scores * repetition_penalty,
                    token_scores / repetition_penalty,
                )
                logits.scatter_(1, input_ids, token_scores)
            if not do_sample:
                next_token = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k > 0:
                    top_k = min(top_k, logits.shape[-1])
                    threshold = torch.topk(logits, top_k, dim=-1).values[:, -1:]
                    logits = logits.masked_fill(logits < threshold, float("-inf"))
                if top_p > 0 and top_p < 1:
                    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                    cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
                    sorted_remove = cumulative_probs > top_p
                    sorted_remove[:, 1:] = sorted_remove[:, :-1].clone()
                    sorted_remove[:, 0] = False
                    remove = torch.zeros_like(logits, dtype=torch.bool)
                    remove.scatter_(1, sorted_indices, sorted_remove)
                    logits = logits.masked_fill(remove, float("-inf"))
                next_token = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1)
            input_ids = torch.cat((input_ids, next_token), dim=1)
    
    output_ids = input_ids
    
    # Decode the whole sequence
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return generated

In [ ]:
# prompt = "Mamba-3 is a state-space model that"

# # Sampling (creative)
# print("=== Sampling ===")
# print(generate_text(prompt, model, tokenizer, max_new_tokens=80, temperature=0.8))

# print("\n=== Greedy ===")
# print(generate_text(prompt, model, tokenizer, max_new_tokens=80, do_sample=False))

In [ ]:
def generate_simple(prompt, model, tokenizer, max_new_tokens=64, temperature=0.7, top_k=50):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
    
    for _ in range(max_new_tokens):
        with torch.inference_mode():
            logits = model(input_ids).logits[:, -1, :]   # full forward every step
        
        if temperature == 0 or top_k == 1:
            next_token = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        
        input_ids = torch.cat([input_ids, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [ ]:
# print(generate_simple("Mamba-3 is a state-space model that", model, tokenizer, max_new_tokens=80))

In [ ]:
# ------------------------------------------------------------
# Helper: simple chunking (you can replace with any splitter)
# ------------------------------------------------------------
def split_corpus(token_ids: torch.Tensor, chunk_size: int = 512, overlap: float = .1) -> List[torch.Tensor]:
    """Split a 1-D token tensor into fixed-size chunks."""
    chunks = []
    for i in range(0, len(token_ids), int(chunk_size*overlap)):
        chunks.append(token_ids[i : i + chunk_size])
    # return torch.nn.utils.rnn.pad_sequence(chunks, batch_first=True)
    return chunks

In [ ]:
# ------------------------------------------------------------
# Core inference / indexing function following the diagram
# ------------------------------------------------------------
@torch.inference_mode()
def index_corpus(
    corpus: str,
    model,
    tokenizer,
    vector_db,                          # any object with .add(ids, vectors)
    chunk_size: int = 512,
    device: str = "cuda",
) -> None:
    """
    Follows the flowchart:

        Start → Tokenize → Split Corpus
                              ↓
                         Next Chunk
                              ↓
                        Update State  →  Extract state to Vector DB
                              ↑                    ↓
                              └──── Corpus Empty? ──┘
                                         │ Yes
                                         ↓
                                        End
    """
    # 1. Tokenize
    token_ids = tokenizer(
        corpus,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids[0].to(device)          # shape: (seq_len,)

    # 2. Split Corpus
    chunks = split_corpus(token_ids, chunk_size=chunk_size)

    vector_db.init_index(max_elements = len(chunks), ef_construction = 200, M = 16)

    embeddings = []
    chunk_data = []

    # 3. Loop: while corpus is not empty
    for chunk_idx, chunk in enumerate(chunks):
        input_ids = chunk.unsqueeze(0)          # (1, chunk_len)

        with torch.no_grad():
            emb = model.backbone(input_ids).squeeze(0)
            state_vector = emb.float().cpu().numpy()

        embeddings.append(state_vector)
        chunk_data.append({"text": tokenizer.decode(chunk, skip_special_tokens=True)})

        if chunk_idx >= 20:
            break

    vector_db.add_items(
        data=embeddings,
        ids=range(len(embeddings)),
    )


In [ ]:
# ------------------------------------------------------------
# Example usage
# ------------------------------------------------------------
vector_db = hnswlib.Index('cosine', 2048)

# corpus = """Your long document or collection of documents goes here..."""
# with open('../input.txt', 'rt') as f:
with open('input.txt', 'rt', encoding='utf-8') as f:
    corpus = f.read()

index_corpus(
    corpus=corpus,
    model=model,
    tokenizer=tokenizer,
    vector_db=vector_db,
    chunk_size=512,
)

print(f"Indexed {vector_db.get_current_count()} chunks")

In [1]:
from ssmrag.rag import *
from datasets import load_dataset
import json

In [2]:
# Main evaluation pipeline
# Load dataset
dataset = load_dataset(
    "framolfese/2WikiMultihopQA",
    split="validation"
)

# Select evaluation questions
eval_dataset = dataset.shuffle(seed=42).select(range(3)) # Adjust the range for more evaluation questions

# Build full validation corpus
documents = load_2wiki_corpus(dataset)

In [3]:
# Initialize model
rag = RAG(
    model_id="state-spaces/mamba3-siso-1.5b",
    tokenizer_id="meta-llama/Llama-3.1-8B",
    device="cuda",
    dtype=torch.bfloat16,
    space="cosine",
    dim=2048
)

In [4]:
# Limit context
s = set()

for i in range(len(eval_dataset)):
    [s.add(title) for title in eval_dataset[i]['context']['title']]

s = list(s)

docs = {t: documents[t] for t in s}

In [5]:
# Ingest corpus
rag.ingest(
    docs,
    strategy="baseline",
    chunk_size=512,
    overlap=0.1
)

(22, 2048)
(22,)
Documents: 22
Chunks: 22


In [7]:
# Evaluate
results = evaluate(
    rag,
    eval_dataset,
    k=5,
    max_new_tokens=100
)

# Save results
with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

Where was the director of film Breakup Buddies born?

        Answer the following question using only the provided context.

        If the context does not contain enough information,
        state that you cannot determine the answer.

        Context:
        Source: Ian Barry (director)
Content: Ian Barry is an Australian director of film and TV.

Source: J. Searle Dawley
Content: James Searle Dawley (May 13, 1877 – March 30, 1949) was an American producer, director and screenwriter. He directed 149 silent films between 1907 and 1926. He was born in Del Norte, Colorado, and died in Hollywood, California. Searley is best remembered for directing the Thomas Edison version of "Frankenstein" in 1910. He reportedly referred to himself as "the first motion picture director", which was not accurate. He worked for a while at Edison's studio, then started his own production company, being quite successful financially, although most of his films are little known today. When critics began to